# NPTEL Study Notes: Introduction to Large Language Models (LLMs)

## Lecture: From Attention to Self-Attention and the Transformer Architecture

---

### Context & Recap

In previous lectures, we explored sequence modeling using RNNs, LSTMs, and GRUs, specifically focusing on **Sequence-to-Sequence (Seq2Seq)** models for tasks like Machine Translation.

* **The Bottleneck Problem:** In vanilla Seq2Seq models, the final hidden state of the encoder must compress the entire input sequence, creating an information bottleneck.
* **Vanilla Attention:** Solved the bottleneck by providing direct connections between the decoder and all encoder states. It calculates a similarity score (dot product) between the decoder state and encoder states, converts these to a probability distribution using **softmax**, and computes a weighted average (attention vector).

**The Remaining Problem:** While attention solves the bottleneck and vanishing gradients, RNNs still process tokens sequentially ($t$ depends on $t-1$). We cannot process tokens in parallel, limiting computational efficiency.

---

### 1. The Information Retrieval Analogy: Query, Key, Value (QKV)

To understand modern attention mechanisms, we must borrow terminology from Information Retrieval (e.g., searching a database).

* **Query (Q):** What you are looking for (e.g., a search term typed by a user).
* **Key (K):** The metadata or descriptors of available items (e.g., keywords or titles of documents).
* **Value (V):** The actual content of the item you retrieve (e.g., the document itself or a specific paragraph).

**In Vanilla Seq2Seq Attention:**

* **Query:** The current hidden state of the Decoder.
* **Key & Value:** The hidden states of the Encoder (Keys and Values are identical in this setup).

---

### 2. Self-Attention Mechanism

**Goal:** Remove recurrent (RNN) connections entirely to allow parallel processing of all tokens while still capturing context.

In **Self-Attention**, a sequence attends to *itself*. For a given sequence of tokens $x_1, x_2, \dots, x_n$, every token acts as a Query, Key, and Value.

#### Step-by-Step Mathematical Process:

1. **Input Embeddings:** Input tokens are passed through a linear layer and non-linearity to obtain hidden state vectors $h_1, h_2, \dots, h_n$.
2. **Generate Q, K, V:** For each token's hidden state $h_i$, we generate three separate vectors using three distinct, learnable parameter matrices: $W_Q, W_K,$ and $W_V$.
* Query: $Q_i = h_i W_Q$
* Key: $K_i = h_i W_K$
* Value: $V_i = h_i W_V$
*(Note: These matrices are shared across all tokens in a sequence, but are unique for each self-attention layer).*


3. **Calculate Attention Scores ($E$):** Compute the dot product between a Query and all Keys in the sequence. For token 1 attending to token 2: $E_{1,2} = Q_1 \cdot K_2$.
4. **Calculate Attention Distribution ($\alpha$):** Pass the scores through a **softmax** function to get a probability distribution (weights that sum to 1).
5. **Compute Attention Vector:** Take the weighted sum of all Values using the attention distribution.
* $Attention\_Vector_1 = \alpha_{1,1}V_1 + \alpha_{1,2}V_2 + \dots + \alpha_{1,n}V_n$



**Complexity:** Because every token attends to every other token, the computational complexity is $O(n^2)$, where $n$ is the sequence length.

---

### 3. Positional Encoding

**The Problem:** Self-attention operations are independent and lack temporal dependency. If you shuffle the input tokens, the output vectors just shuffle accordingly (the model is **position-invariant**). "The man eats cheese" and "Cheese eats the man" would be processed identically.

**The Solution:** Inject positional information directly into the input embeddings before feeding them into the self-attention block.

* Let $E_i$ be the token embedding (e.g., from Word2Vec/GloVe or one-hot encoding).
* Let $P_i$ be a unique Positional Encoding vector for position $i$.
* **New Input** = $E_i + P_i$ (Element-wise addition).
*(Note: The exact mathematical formulation of $P_i$ will be covered in a subsequent lecture).*

---

### 4. Multi-Headed Self-Attention

Why use one set of $W_Q, W_K, W_V$ when we can use many? This is inspired by **Convolutional Neural Networks (CNNs)**, where multiple filters/kernels are applied to an image to capture different visual features (edges, textures, eyes, ears).

In NLP, different **"Attention Heads"** can focus on different linguistic aspects of a token simultaneously:

* Head 1 might focus on syntax (e.g., identifying if the token is a subject or object).
* Head 2 might focus on POS tagging (e.g., is it a noun or pronoun?).
* Head 3 might focus on Named Entity Recognition (NER).

#### Mechanism:

1. Assume we have $H$ attention heads. We instantiate $H$ sets of weight matrices: $(W_Q^1, W_K^1, W_V^1), \dots, (W_Q^H, W_K^H, W_V^H)$.
2. Each head processes the input independently and produces its own attention vector for a token of dimension $d_{v}$.
3. **Concatenation:** For a single token, concatenate the vectors from all heads. If we have 2 heads and the original dimension is $d$, the concatenated vector size is $2d$.
4. **Linear Projection:** Pass the concatenated vector through a final linear output layer ($W_O$) to project it back down to the original dimension $d$, ensuring it is ready for the next layer.

---

### 5. Position-wise Feed-Forward Network (FFN)

**The Linearity Problem:** If you look closely at self-attention, despite the softmax, the final operation is a weighted sum of hidden states. It is fundamentally a linear combination. To learn complex patterns, neural networks require strong non-linearity.

**The Solution:** Add a Feed-Forward Network (FFN) after the self-attention block, applied to each token's vector *independently* and *identically* (hence "position-wise").

#### Mechanism & Analogy:

* **Self-Attention = Memory Fetching:** Message passing. Tokens exchange context and "fetch" information from each other.
* **FFN = Actual Processing:** Applies non-linear transformations to the fetched contextual information.
* The FFN typically consists of two linear transformations with a ReLU (or GELU) activation in between.
* **Dimensionality:** The FFN usually projects the $d$-dimensional vector into a higher dimensional space (e.g., $4d$) to process it extensively, and then projects it back down to $d$.

---

### 6. Masked Self-Attention (For Decoders)

During training in an Encoder-Decoder translation task, we have access to the full target sentence. However, during inference (testing), the Decoder generates words autoregressively (one by one) and cannot look into the future.

If we use standard self-attention in the Decoder during training, it would "cheat" by looking ahead at words it hasn't predicted yet.

#### The Masking Trick:

To prevent a query at position $t$ ($Q_t$) from attending to keys at future positions ($K_{t+1}, K_{t+2}$):

1. Compute the standard $Q \times K^T$ matrix.
2. Apply a mask: Set all upper-diagonal elements (representing future token connections) to $-\infty$.
3. Apply Softmax: Because $e^{-\infty} = 0$, the softmax probabilities for all future tokens become exactly 0.
4. Result: The model learns to predict token $t+1$ using only context from tokens $1$ to $t$.

---

### 7. Cross-Attention

How does the Decoder look at the Encoder's output? Through **Cross-Attention**.

In this specific attention layer inside the Decoder:

* **Query (Q):** Comes from the current state of the **Decoder** (after passing through masked self-attention).
* **Key (K) and Value (V):** Come from the final output states of the **Encoder**.
* **Masking?** No masking is required here because the entire source sentence (Encoder) is already available, and Decoder states are not attending to other Decoder states in this specific sub-layer.

---

### 8. The Transformer Architecture Summary

Based on the lecture, a Transformer model comprises stacks of Encoder and Decoder blocks.

**Encoder Block contains:**

1. Unmasked Multi-Headed Self-Attention
2. Position-wise Feed-Forward Network

**Decoder Block contains:**

1. Masked Multi-Headed Self-Attention (prevents looking ahead)
2. Cross-Attention (attends to Encoder)
3. Position-wise Feed-Forward Network

*(Instructor's Note: Components like Residual Connections and "Add & Norm" (Layer Normalization) blocks wrap around these layers, which will be detailed in the next lecture).*

---

### 📝 NPTEL Practice Questions

**Q1. In the context of the self-attention mechanism, how are the Query, Key, and Value vectors generated for a specific token?**
a) They are randomly initialized and kept static during training.
b) By multiplying the token's hidden state with three separate learnable parameter matrices ($W_Q, W_K, W_V$).
c) The Query is generated from the decoder, while Key and Value are generated from the encoder.
d) By taking the dot product of the token's embedding with the positional encoding vector.

* **Answer:** (b)
* **Explanation:** In self-attention, the Query, Key, and Value are derived linearly from the same hidden state by multiplying it with three distinct learned weight matrices. Option (c) describes Cross-Attention, not Self-Attention.

**Q2. Why is "Masked" Self-Attention necessary in the Decoder block during the training phase?**
a) To reduce the $O(n^2)$ computational complexity of the attention matrix.
b) To prevent the vanishing gradient problem in deep networks.
c) To prevent the model from "cheating" by attending to future tokens that would not be available during inference.
d) To filter out stop words and irrelevant punctuation from the target sentence.

* **Answer:** (c)
* **Explanation:** During autoregressive generation at test time, future tokens do not exist yet. Masking the upper diagonal elements with $-\infty$ during training ensures the softmax assigns a probability of 0 to future tokens, forcing the model to rely only on past context.

**Q3. What is the primary role of the Position-wise Feed-Forward Network (FFN) in a Transformer block?**
a) To provide absolute positional information to the tokens.
b) To connect the decoder states directly to the encoder states.
c) To add necessary non-linearity and process the contextual "memory" fetched by the self-attention layer.
d) To convert the final hidden states into a probability distribution over the vocabulary.

* **Answer:** (c)
* **Explanation:** Self-attention is largely a linear operation (a weighted sum). The FFN introduces non-linearity (e.g., via ReLU) to learn complex representations from the aggregated attention vectors.

**Q4. If a Transformer model utilizes 8 Attention Heads, and the original embedding dimension $d$ is 512, what happens immediately after the 8 individual attention vectors are calculated for a token?**
a) They are averaged element-wise to form a single 512-dimensional vector.
b) They are multiplied together to produce attention scores.
c) They are concatenated and then passed through a linear projection matrix to return the dimension to 512.
d) They are passed directly into the Cross-Attention layer without modification.

* **Answer:** (c)
* **Explanation:** The outputs of multiple heads are concatenated (creating a wider vector, e.g., $8 \times d_v$) and then multiplied by an output weight matrix ($W_O$) to project the data back to the expected dimension $d$ for the next layer.

**Q5. In the Information Retrieval analogy used to explain Attention, what does the "Value" correspond to?**
a) The search string typed by the user.
b) The metadata or keywords of a document.
c) The actual content of the retrieved document or paragraph.
d) The similarity score calculated by the dot product.

* **Answer:** (c)
* **Explanation:** Query is the search term, Key is the document metadata, and Value is the document content itself that is returned based on the Query-Key match.

---

### 🚀 Quick Revision Summary

* **Self-Attention:** Removes RNN sequential bottlenecks. Every token calculates $Q, K, V$ and attends to every other token in parallel. Complexity is $O(n^2)$.
* **Positional Encoding:** Added to input embeddings because self-attention has no inherent sense of sequence order.
* **Multi-Head Attention:** Multiple $W_Q, W_K, W_V$ sets allow the model to focus on different linguistic aspects (syntax, semantics) simultaneously.
* **Feed-Forward Network (FFN):** Acts as the non-linear processing unit after attention aggregates the context. Usually features an up-projection followed by a down-projection.
* **Masking:** Upper diagonal of the Q-K dot product matrix is set to $-\infty$ in the decoder to hide future tokens during autoregressive training.
* **Cross-Attention:** Links the two blocks. Decoder provides Queries; Encoder provides Keys and Values. No masking is used here.